In [1]:
import pymongo
from collections import defaultdict

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# Create the unique index on organisasjonsnummer in the companies
# collection. Unique because load_companies.py merges on that field,
# and because it is what stops a stray plain mongoimport from silently
# duplicating the register.
# One-time operation - persists going forward. In MongoDB 4.2+,
# index builds run in the background and don't block your fetch
# script's writes to financial_data, so this is safe to run now.
# ============================================================
companies_col.create_index("organisasjonsnummer", unique=True)
print("Index created (or already existed).")

# ============================================================
# Pull a limited sample from financial_data instead of the whole
# collection. Note: this is "the first 10,000 records written so
# far", not a statistically random sample - worth keeping in mind
# when interpreting percentages.
# ============================================================
SAMPLE_SIZE = 10000

financial_sample = list(financial_col.find(
    {},
    {"organisasjonsnummer": 1, "fetch_status": 1, "_id": 0}
).limit(SAMPLE_SIZE))

print(f"Analyzing {len(financial_sample)} fetched records")

org_numbers_in_sample = [d["organisasjonsnummer"] for d in financial_sample]

# ============================================================
# ONE indexed batch query instead of a per-document $lookup join
# ============================================================
company_lookup = {
    c["organisasjonsnummer"]: c
    for c in companies_col.find(
        {"organisasjonsnummer": {"$in": org_numbers_in_sample}},
        {"organisasjonsnummer": 1, "organisasjonsform.beskrivelse": 1,
         "stiftelsesdato": 1, "konkurs": 1, "underAvvikling": 1,
         "registrertIForetaksregisteret": 1, "navn": 1}
    )
}

# ============================================================
# Breakdown by organisasjonsform - joining in Python, not MongoDB
# ============================================================
breakdown = defaultdict(lambda: {"success": 0, "no_data": 0})

for record in financial_sample:
    org = record["organisasjonsnummer"]
    status = record["fetch_status"]
    company = company_lookup.get(org)
    form = company["organisasjonsform"]["beskrivelse"] if company and company.get("organisasjonsform") else "UNKNOWN"
    breakdown[form][status] += 1

print(f"\n{'Organisasjonsform':<40} {'Success':>8} {'No data':>8} {'Total':>8} {'No-data %':>10}")
print("-" * 78)
for form, counts in sorted(breakdown.items(), key=lambda x: -(x[1]['success'] + x[1]['no_data'])):
    total = counts['success'] + counts['no_data']
    pct = counts['no_data'] / total * 100 if total > 0 else 0
    print(f"{str(form):<40} {counts['success']:>8} {counts['no_data']:>8} {total:>8} {pct:>9.1f}%")

# ============================================================
# 10 concrete no_data examples, in full
# ============================================================
print("\n=== Sample of actual NO_DATA companies ===")
no_data_orgs = [d["organisasjonsnummer"] for d in financial_sample if d["fetch_status"] == "no_data"][:10]
for org in no_data_orgs:
    c = company_lookup.get(org)
    if c:
        print(f"{c['organisasjonsnummer']} | {c['navn']} | "
              f"form={c.get('organisasjonsform', {}).get('beskrivelse')} | "
              f"stiftet={c.get('stiftelsesdato')} | "
              f"konkurs={c.get('konkurs')} | "
              f"underAvvikling={c.get('underAvvikling')} | "
              f"registrertIForetaksregisteret={c.get('registrertIForetaksregisteret')}")

Index created (or already existed).
Analyzing 10000 fetched records

Organisasjonsform                         Success  No data    Total  No-data %
------------------------------------------------------------------------------
Enkeltpersonforetak                            20     3910     3930      99.5%
Aksjeselskap                                 3503      201     3704       5.4%
Forening/lag/innretning                        25     1058     1083      97.7%
Eierseksjonssameie                             91      201      292      68.8%
Utenlandsk enhet                                0      253      253     100.0%
Norskregistrert utenlandsk foretak             41      182      223      81.6%
Ansvarlig selskap med delt ansvar               6       86       92      93.5%
Borettslag                                     90        1       91       1.1%
Tingsrettslig sameie                           10       62       72      86.1%
Ansvarlig selskap med solidarisk ansvar         2       60    

In [2]:
import pymongo
from collections import Counter

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
financial_col = db["financial_data"]

# ============================================================
# Aggregation runs on the MongoDB server, not pulled into Python -
# efficient even as financial_data keeps growing during the fetch.
# Unwinds the "data" array (in case any company has multiple
# periods) and extracts the year from each period's end date.
# ============================================================
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "year": {"$substrBytes": ["$data.regnskapsperiode.tilDato", 0, 4]}
    }},
    {"$group": {
        "_id": "$year",
        "count": {"$sum": 1}
    }},
    {"$sort": {"_id": -1}}
]

results = list(financial_col.aggregate(pipeline))

print(f"{'Fiscal year (tilDato)':<25} {'Count':>10}")
print("-" * 36)
total = 0
for r in results:
    print(f"{r['_id']:<25} {r['count']:>10}")
    total += r['count']
print("-" * 36)
print(f"{'Total period-entries':<25} {total:>10}")

# ============================================================
# Check whether any single company has more than one period
# in "data" - i.e. whether multi-year data is actually present
# per company, not just varying latest-year across companies
# ============================================================
multi_period = financial_col.count_documents({
    "fetch_status": "success",
    "data.1": {"$exists": True}  # true only if index 1 exists, i.e. 2+ entries
})
single_period = financial_col.count_documents({
    "fetch_status": "success",
    "data.1": {"$exists": False}
})

print(f"\nCompanies with exactly 1 period: {single_period:,}")
print(f"Companies with 2+ periods:       {multi_period:,}")

Fiscal year (tilDato)          Count
------------------------------------
2026                             183
2025                          417897
2024                           19133
2023                            3522
2022                            1341
2021                             746
2020                             540
2019                             468
2018                             408
2017                             385
2016                              13
2015                               4
2014                               4
2013                               1
2011                               1
------------------------------------
Total period-entries          444646

Companies with exactly 1 period: 444,646
Companies with 2+ periods:       0


In [3]:
import pymongo

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# Step 1: get org number + latest filed year for all successful fetches
# ============================================================
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "organisasjonsnummer": 1,
        "year": {"$substrBytes": ["$data.regnskapsperiode.tilDato", 0, 4]}
    }}
]
records = list(financial_col.aggregate(pipeline))
# The Regnskapsregisteret API returns only the latest available filing, so
# "data" normally holds a single accounting period and this collapses to one
# entry per company. max() rather than the last unwound record makes the
# intent explicit: where several periods do exist, the most recent is used.
year_by_org = {}
for r in records:
    org = r["organisasjonsnummer"]
    year_by_org[org] = max(year_by_org.get(org, ""), r["year"])

# Split into "recent" (2025+) vs "old" (<2025) groups
recent_orgs = [org for org, year in year_by_org.items() if year >= "2025"]
old_orgs = [org for org, year in year_by_org.items() if year < "2025"]

print(f"Recent (2025+): {len(recent_orgs)}")
print(f"Old (<2025):    {len(old_orgs)}")

# ============================================================
# Step 2: look up status flags via indexed $in query (not $lookup)
# ============================================================
fields = {"organisasjonsnummer": 1, "konkurs": 1, "underAvvikling": 1,
          "underTvangsavviklingEllerTvangsopplosning": 1,
          "sisteInnsendteAarsregnskap": 1, "_id": 0}

recent_companies = list(companies_col.find({"organisasjonsnummer": {"$in": recent_orgs}}, fields))
old_companies = list(companies_col.find({"organisasjonsnummer": {"$in": old_orgs}}, fields))

def summarize(companies, label):
    n = len(companies)
    konkurs = sum(1 for c in companies if c.get("konkurs"))
    avvikling = sum(1 for c in companies if c.get("underAvvikling"))
    tvang = sum(1 for c in companies if c.get("underTvangsavviklingEllerTvangsopplosning"))
    any_flag = sum(1 for c in companies if c.get("konkurs") or c.get("underAvvikling") or c.get("underTvangsavviklingEllerTvangsopplosning"))
    print(f"\n=== {label} (n={n}) ===")
    print(f"  konkurs=True:                              {konkurs} ({konkurs/n*100:.1f}%)")
    print(f"  underAvvikling=True:                        {avvikling} ({avvikling/n*100:.1f}%)")
    print(f"  underTvangsavviklingEllerTvangsopplosning:  {tvang} ({tvang/n*100:.1f}%)")
    print(f"  any of the above flags:                     {any_flag} ({any_flag/n*100:.1f}%)")

summarize(recent_companies, "RECENT filings (2025+)")
summarize(old_companies, "OLD filings (<2025)")

# ============================================================
# Step 3: cross-check regnskapsregisteret year vs Brreg's own
# sisteInnsendteAarsregnskap field, for consistency
# ============================================================
old_lookup = {c["organisasjonsnummer"]: c for c in old_companies}
match, mismatch = 0, 0
for org in old_orgs:
    c = old_lookup.get(org)
    if c and c.get("sisteInnsendteAarsregnskap") == year_by_org[org]:
        match += 1
    else:
        mismatch += 1
print(f"\nOld group: sisteInnsendteAarsregnskap matches API year: {match}, mismatches: {mismatch}")

# ============================================================
# Step 4: concrete examples from the old group
# ============================================================
print("\n=== Sample of OLD-filing companies with their status ===")
for org in old_orgs[:15]:
    c = old_lookup.get(org)
    if c:
        print(f"{org} | api_year={year_by_org[org]} | konkurs={c.get('konkurs')} | "
              f"underAvvikling={c.get('underAvvikling')} | "
              f"tvang={c.get('underTvangsavviklingEllerTvangsopplosning')} | "
              f"sisteInnsendteAarsregnskap={c.get('sisteInnsendteAarsregnskap')}")

Recent (2025+): 418080
Old (<2025):    26566

=== RECENT filings (2025+) (n=418080) ===
  konkurs=True:                              273 (0.1%)
  underAvvikling=True:                        4434 (1.1%)
  underTvangsavviklingEllerTvangsopplosning:  78 (0.0%)
  any of the above flags:                     4762 (1.1%)

=== OLD filings (<2025) (n=26566) ===
  konkurs=True:                              2632 (9.9%)
  underAvvikling=True:                        734 (2.8%)
  underTvangsavviklingEllerTvangsopplosning:  1489 (5.6%)
  any of the above flags:                     4707 (17.7%)

Old group: sisteInnsendteAarsregnskap matches API year: 26322, mismatches: 244

=== Sample of OLD-filing companies with their status ===
824791252 | api_year=2023 | konkurs=True | underAvvikling=False | tvang=False | sisteInnsendteAarsregnskap=2023
928038831 | api_year=2024 | konkurs=False | underAvvikling=False | tvang=False | sisteInnsendteAarsregnskap=2024
917332487 | api_year=2021 | konkurs=False | underAv

In [4]:
import pymongo
from collections import Counter

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# Step 1: same recent/old split as before
# ============================================================
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "organisasjonsnummer": 1,
        "year": {"$substrBytes": ["$data.regnskapsperiode.tilDato", 0, 4]}
    }}
]
records = list(financial_col.aggregate(pipeline))
# The Regnskapsregisteret API returns only the latest available filing, so
# "data" normally holds a single accounting period and this collapses to one
# entry per company. max() rather than the last unwound record makes the
# intent explicit: where several periods do exist, the most recent is used.
year_by_org = {}
for r in records:
    org = r["organisasjonsnummer"]
    year_by_org[org] = max(year_by_org.get(org, ""), r["year"])

recent_orgs = [org for org, year in year_by_org.items() if year >= "2025"]
old_orgs = [org for org, year in year_by_org.items() if year < "2025"]

print(f"Recent (2025+): {len(recent_orgs)}")
print(f"Old (<2025):    {len(old_orgs)}")

# ============================================================
# Step 2: pull a broader set of candidate fields for both groups
# ============================================================
fields = {
    "organisasjonsnummer": 1,
    "organisasjonsform.beskrivelse": 1,
    "registrertIMvaregisteret": 1,
    "harRegistrertAntallAnsatte": 1,
    "erIKonsern": 1,
    "registrertIForetaksregisteret": 1,
    "stiftelsesdato": 1,
    "kapital.belop": 1,
    "_id": 0
}

recent_companies = list(companies_col.find({"organisasjonsnummer": {"$in": recent_orgs}}, fields))
old_companies = list(companies_col.find({"organisasjonsnummer": {"$in": old_orgs}}, fields))

# ============================================================
# Step 3: compare candidate factors side by side
# ============================================================
def rate(companies, predicate, label):
    n = len(companies)
    true_count = sum(1 for c in companies if predicate(c))
    pct = true_count / n * 100 if n > 0 else 0
    print(f"  {label:<40} {true_count:>6} / {n:<6} ({pct:>5.1f}%)")

def summarize_group(companies, label):
    n = len(companies)
    print(f"\n=== {label} (n={n}) ===")
    rate(companies, lambda c: c.get("registrertIMvaregisteret") is True, "VAT registered")
    rate(companies, lambda c: c.get("harRegistrertAntallAnsatte") is True, "Has registered employees")
    rate(companies, lambda c: c.get("erIKonsern") is True, "Part of corporate group")
    rate(companies, lambda c: c.get("registrertIForetaksregisteret") is True, "Registered in Foretaksregisteret")
    rate(companies, lambda c: (c.get("kapital") or {}).get("belop") is not None, "Share capital on file")

    # Top 5 organisasjonsform within this group
    forms = Counter(c.get("organisasjonsform", {}).get("beskrivelse", "UNKNOWN") for c in companies)
    print(f"  Top organisasjonsform:")
    for form, count in forms.most_common(5):
        print(f"    {form:<38} {count:>6} ({count/n*100:.1f}%)")

    # Median company age AT THE TIME of its latest filing (not today)
    ages = []
    for c in companies:
        org = c["organisasjonsnummer"]
        stiftet = c.get("stiftelsesdato")
        filing_year = year_by_org.get(org)
        if stiftet and filing_year:
            try:
                age = int(filing_year) - int(stiftet[:4])
                ages.append(age)
            except (ValueError, TypeError):
                pass
    if ages:
        ages.sort()
        median_age = ages[len(ages) // 2]
        print(f"  Median company age at time of last filing: {median_age} years (n={len(ages)})")

summarize_group(recent_companies, "RECENT filings (2025+)")
summarize_group(old_companies, "OLD filings (<2025)")

Recent (2025+): 418080
Old (<2025):    26566

=== RECENT filings (2025+) (n=418080) ===
  VAT registered                           214261 / 418080 ( 51.2%)
  Has registered employees                 145013 / 418080 ( 34.7%)
  Part of corporate group                   27675 / 418080 (  6.6%)
  Registered in Foretaksregisteret         408783 / 418080 ( 97.8%)
  Share capital on file                    389978 / 418080 ( 93.3%)
  Top organisasjonsform:
    Aksjeselskap                           384536 (92.0%)
    Borettslag                               9900 (2.4%)
    Eierseksjonssameie                       8635 (2.1%)
    Stiftelse                                5272 (1.3%)
    Norskregistrert utenlandsk foretak       2553 (0.6%)
  Median company age at time of last filing: 8 years (n=413924)

=== OLD filings (<2025) (n=26566) ===
  VAT registered                            10629 / 26566  ( 40.0%)
  Has registered employees                   6710 / 26566  ( 25.3%)
  Part of corporate gr

In [5]:
import pymongo

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# Recompute the "old" group (API returned year < 2025)
# ============================================================
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "organisasjonsnummer": 1,
        "api_year": {"$substrBytes": ["$data.regnskapsperiode.tilDato", 0, 4]}
    }}
]
records = list(financial_col.aggregate(pipeline))
# The Regnskapsregisteret API returns only the latest available filing, so
# "data" normally holds a single accounting period and this collapses to one
# entry per company. max() rather than the last unwound record makes the
# intent explicit: where several periods do exist, the most recent is used.
api_year_by_org = {}
for r in records:
    org = r["organisasjonsnummer"]
    api_year_by_org[org] = max(api_year_by_org.get(org, ""), r["api_year"])

# Derived from the dict, so each company appears once.
old_orgs = [org for org, year in api_year_by_org.items() if year < "2025"]

print(f"Old group (API year < 2025): {len(old_orgs)}")

# ============================================================
# Fetch the exact fields needed for this specific combination
# ============================================================
fields = {
    "organisasjonsnummer": 1, "navn": 1,
    "sisteInnsendteAarsregnskap": 1,
    "konkurs": 1, "underAvvikling": 1,
    "underTvangsavviklingEllerTvangsopplosning": 1,
    "_id": 0
}
old_companies = list(companies_col.find({"organisasjonsnummer": {"$in": old_orgs}}, fields))

# ============================================================
# Filter for the EXACT combination: Brreg says filed 2025,
# but none of the three distress flags are set
# ============================================================
unexplained = [
    c for c in old_companies
    if c.get("sisteInnsendteAarsregnskap") == "2025"
    and c.get("konkurs") is False
    and c.get("underAvvikling") is False
    and c.get("underTvangsavviklingEllerTvangsopplosning") is False
]

print(f"\nOld-group companies where Brreg says sisteInnsendteAarsregnskap=2025,")
print(f"AND konkurs=False, underAvvikling=False, underTvangsavviklingEllerTvangsopplosning=False:")
print(f"  {len(unexplained)} / {len(old_orgs)} ({len(unexplained)/len(old_orgs)*100:.1f}%)")

print(f"\n=== Sample of these unexplained cases ===")
for c in unexplained[:15]:
    org = c["organisasjonsnummer"]
    print(f"{org} | {c.get('navn')} | api_year={api_year_by_org[org]} | "
          f"sisteInnsendteAarsregnskap={c.get('sisteInnsendteAarsregnskap')} | "
          f"konkurs={c.get('konkurs')} | underAvvikling={c.get('underAvvikling')} | "
          f"tvang={c.get('underTvangsavviklingEllerTvangsopplosning')}")

Old group (API year < 2025): 26566

Old-group companies where Brreg says sisteInnsendteAarsregnskap=2025,
AND konkurs=False, underAvvikling=False, underTvangsavviklingEllerTvangsopplosning=False:
  113 / 26566 (0.4%)

=== Sample of these unexplained cases ===
817201202 | SKØYTEKLUBBEN LØRENSKOG | api_year=2022 | sisteInnsendteAarsregnskap=2025 | konkurs=False | underAvvikling=False | tvang=False
822061532 | FOTBALL ANYONE? | api_year=2019 | sisteInnsendteAarsregnskap=2025 | konkurs=False | underAvvikling=False | tvang=False
831214872 | CARRHAE CAPITAL LLP NUF | api_year=2024 | sisteInnsendteAarsregnskap=2025 | konkurs=False | underAvvikling=False | tvang=False
870009852 | ALTA HISTORIELAG | api_year=2024 | sisteInnsendteAarsregnskap=2025 | konkurs=False | underAvvikling=False | tvang=False
871545472 | FORENINGEN FOR OMPLASSERING AV DYR FOD | api_year=2020 | sisteInnsendteAarsregnskap=2025 | konkurs=False | underAvvikling=False | tvang=False
875647792 | NOTODDEN RØDE KORS | api_year=202

In [6]:
import pymongo
from datetime import datetime, timezone

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

CURRENT_YEAR = datetime.now(timezone.utc).year

# ============================================================
# Step 1: recompute the old group (api_year < 2025)
# ============================================================
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "organisasjonsnummer": 1,
        "api_year": {"$substrBytes": ["$data.regnskapsperiode.tilDato", 0, 4]}
    }}
]
records = list(financial_col.aggregate(pipeline))
# The Regnskapsregisteret API returns only the latest available filing, so
# "data" normally holds a single accounting period and this collapses to one
# entry per company. max() rather than the last unwound record makes the
# intent explicit: where several periods do exist, the most recent is used.
api_year_by_org = {}
for r in records:
    org = r["organisasjonsnummer"]
    api_year_by_org[org] = max(api_year_by_org.get(org, ""), r["api_year"])
old_orgs = [org for org, year in api_year_by_org.items() if year < "2025"]

print(f"Old group (api_year < 2025): {len(old_orgs)}")

# ============================================================
# Step 2: pull founding date + distress flags (indexed lookup)
# ============================================================
fields = {
    "organisasjonsnummer": 1, "navn": 1, "stiftelsesdato": 1,
    "konkurs": 1, "underAvvikling": 1,
    "underTvangsavviklingEllerTvangsopplosning": 1,
    "_id": 0
}
old_companies = list(companies_col.find({"organisasjonsnummer": {"$in": old_orgs}}, fields))

# ============================================================
# Step 3: split by company age NOW (not age at filing)
# ============================================================
RECENT_FOUNDING_THRESHOLD_YEARS = 2

recent_founded, established, no_founding_date = [], [], []

for c in old_companies:
    org = c["organisasjonsnummer"]
    stiftet = c.get("stiftelsesdato")
    if not stiftet:
        no_founding_date.append(c)
        continue
    founding_year = int(stiftet[:4])
    age_now = CURRENT_YEAR - founding_year
    api_year = int(api_year_by_org[org])
    gap_founding_to_filing = api_year - founding_year  # how soon after founding was their (only) filing?

    record = {**c, "age_now": age_now, "api_year": api_year, "gap_founding_to_filing": gap_founding_to_filing}
    if age_now <= RECENT_FOUNDING_THRESHOLD_YEARS:
        recent_founded.append(record)
    else:
        established.append(record)

print(f"\nOld group breakdown:")
print(f"  Recently founded (<= {RECENT_FOUNDING_THRESHOLD_YEARS}y old, now): {len(recent_founded)}")
print(f"  Established (> {RECENT_FOUNDING_THRESHOLD_YEARS}y old):           {len(established)}")
print(f"  No founding date on file:                   {len(no_founding_date)}")

# ============================================================
# Step 4: is their one filing close to founding (first-ever filing)?
# ============================================================
if recent_founded:
    first_filing_like = sum(1 for r in recent_founded if r["gap_founding_to_filing"] <= 1)
    print(f"\nOf recently-founded old-filers, {first_filing_like}/{len(recent_founded)} "
          f"have their (only) filing within 1 year of founding")
    print("  -> consistent with: first-ever filing, simply not caught up to the next one yet")

# ============================================================
# Step 5: distress-flag rate, recent vs established
# ============================================================
def distress_rate(group, label):
    n = len(group)
    if n == 0:
        print(f"  {label}: n=0, skipping")
        return
    flagged = sum(1 for r in group if r.get("konkurs") or r.get("underAvvikling") or r.get("underTvangsavviklingEllerTvangsopplosning"))
    print(f"  {label:<25} {flagged}/{n} ({flagged/n*100:.1f}%) have a distress flag")

print(f"\nDistress flag rate by subgroup:")
distress_rate(recent_founded, "Recently founded")
distress_rate(established, "Established")

# ============================================================
# Step 6: concrete examples
# ============================================================
print(f"\n=== Sample of recently-founded old-filers ===")
for r in recent_founded[:10]:
    print(f"{r['organisasjonsnummer']} | {r['navn']} | founded={r.get('stiftelsesdato')} | "
          f"api_year={r['api_year']} | age_now={r['age_now']} | "
          f"konkurs={r.get('konkurs')} | underAvvikling={r.get('underAvvikling')}")

Old group (api_year < 2025): 26566

Old group breakdown:
  Recently founded (<= 2y old, now): 1234
  Established (> 2y old):           22470
  No founding date on file:                   2862

Of recently-founded old-filers, 1234/1234 have their (only) filing within 1 year of founding
  -> consistent with: first-ever filing, simply not caught up to the next one yet

Distress flag rate by subgroup:
  Recently founded          181/1234 (14.7%) have a distress flag
  Established               4516/22470 (20.1%) have a distress flag

=== Sample of recently-founded old-filers ===
832832642 | FREDRIKSEN LEGETJENESTER AS | founded=2024-01-01 | api_year=2024 | age_now=2 | konkurs=False | underAvvikling=False
832873632 | COREWEAVE NORWAY AS | founded=2024-01-15 | api_year=2024 | age_now=2 | konkurs=False | underAvvikling=False
832912522 | ELISOF AS | founded=2024-01-02 | api_year=2024 | age_now=2 | konkurs=False | underAvvikling=False
832918482 | NW PROPERTIES AS | founded=2024-01-01 | api_year

In [7]:
import pymongo
from datetime import datetime, timezone

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

print("=" * 60)
print("FINANCIAL DATA FETCH - STATUS REPORT")
print(f"Generated: {datetime.now(timezone.utc).isoformat()}")
print("=" * 60)

# ============================================================
# Overall progress
# ============================================================
total_companies = companies_col.count_documents({})
total_fetched = financial_col.count_documents({})
success_count = financial_col.count_documents({"fetch_status": "success"})
no_data_count = financial_col.count_documents({"fetch_status": "no_data"})
remaining = total_companies - total_fetched

print(f"\n--- OVERALL PROGRESS ---")
print(f"Total companies:   {total_companies:,}")
print(f"Processed:         {total_fetched:,} ({total_fetched/total_companies*100:.2f}%)")
print(f"  - success:       {success_count:,} ({success_count/total_fetched*100:.1f}% of processed)" if total_fetched else "  - success: 0")
print(f"  - no_data:       {no_data_count:,} ({no_data_count/total_fetched*100:.1f}% of processed)" if total_fetched else "  - no_data: 0")
print(f"Remaining:         {remaining:,} ({remaining/total_companies*100:.2f}%)")

# ============================================================
# Fetch rate - lifetime average and recent (last 10 min)
# ============================================================
first_doc = financial_col.find_one(sort=[("fetched_at", 1)])
last_doc = financial_col.find_one(sort=[("fetched_at", -1)])

if first_doc and last_doc:
    first_time, last_time = first_doc["fetched_at"], last_doc["fetched_at"]
    elapsed_seconds = (last_time - first_time).total_seconds()
    overall_rate = total_fetched / elapsed_seconds if elapsed_seconds > 0 else 0

    print(f"\n--- FETCH RATE (lifetime average) ---")
    print(f"First fetch:       {first_time.isoformat()}")
    print(f"Most recent fetch: {last_time.isoformat()}")
    print(f"Elapsed:           {elapsed_seconds/3600:.1f} hours")
    print(f"Average rate:      {overall_rate:.2f} companies/sec ({overall_rate*3600:.0f}/hour)")

    ten_min_ago = datetime.fromtimestamp(datetime.now(timezone.utc).timestamp() - 600, tz=timezone.utc)
    recent_count = financial_col.count_documents({"fetched_at": {"$gte": ten_min_ago}})
    recent_rate = recent_count / 600 if recent_count > 0 else 0

    print(f"\n--- RECENT ACTIVITY (last 10 min) ---")
    print(f"Fetched in last 10 min: {recent_count:,}")
    print(f"Recent rate:            {recent_rate:.2f} companies/sec")
    if recent_rate > 0:
        eta_hours = (remaining / recent_rate) / 3600
        print(f"ETA at recent rate:     {eta_hours:.1f} hours ({eta_hours/24:.1f} days)")
    else:
        print("No activity in the last 10 minutes - script may not be running.")
else:
    print("\nNo fetched data yet.")

# ============================================================
# Coverage by organisasjonsform (indexed $lookup - efficient)
# ============================================================
print(f"\n--- COVERAGE BY ORGANISASJONSFORM (top 10 by total count) ---")

coverage_pipeline = [
    {"$lookup": {
        "from": "companies",
        "localField": "organisasjonsnummer",
        "foreignField": "organisasjonsnummer",
        "as": "company"
    }},
    {"$unwind": "$company"},
    {"$group": {"_id": "$company.organisasjonsform.beskrivelse", "fetched": {"$sum": 1}}}
]
fetched_by_form = {r["_id"]: r["fetched"] for r in financial_col.aggregate(coverage_pipeline)}

total_by_form = {
    r["_id"]: r["total"] for r in companies_col.aggregate([
        {"$group": {"_id": "$organisasjonsform.beskrivelse", "total": {"$sum": 1}}}
    ])
}

print(f"{'Organisasjonsform':<38} {'Fetched':>10} {'Total':>10} {'Coverage':>10}")
print("-" * 70)
for form, total in sorted(total_by_form.items(), key=lambda x: -x[1])[:10]:
    fetched = fetched_by_form.get(form, 0)
    pct = fetched / total * 100 if total > 0 else 0
    print(f"{str(form):<38} {fetched:>10,} {total:>10,} {pct:>9.1f}%")

print("\n" + "=" * 60)

FINANCIAL DATA FETCH - STATUS REPORT
Generated: 2026-09-07T19:45:04.456387+00:00

--- OVERALL PROGRESS ---
Total companies:   1,171,373
Processed:         1,170,292 (99.91%)
  - success:       444,646 (38.0% of processed)
  - no_data:       725,646 (62.0% of processed)
Remaining:         1,081 (0.09%)

--- FETCH RATE (lifetime average) ---
First fetch:       2026-08-27T14:12:21.071000
Most recent fetch: 2026-09-03T12:40:15.842000
Elapsed:           166.5 hours
Average rate:      1.95 companies/sec (7030/hour)

--- RECENT ACTIVITY (last 10 min) ---
Fetched in last 10 min: 0
Recent rate:            0.00 companies/sec
No activity in the last 10 minutes - script may not be running.

--- COVERAGE BY ORGANISASJONSFORM (top 10 by total count) ---
Organisasjonsform                         Fetched      Total   Coverage
----------------------------------------------------------------------
Enkeltpersonforetak                       461,121    461,154     100.0%
Aksjeselskap                       

In [8]:
import pymongo

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"

def find_no_data_example(organisasjonsform_kode):
    """
    Finds one company with the given organisasjonsform (e.g. 'AS' or 'ENK')
    that has fetch_status='no_data', by first getting no_data org numbers,
    then filtering by form using the indexed companies collection.
    """
    no_data_orgs = [d["organisasjonsnummer"] for d in financial_col.find(
        {"fetch_status": "no_data"}, {"organisasjonsnummer": 1, "_id": 0}
    ).limit(5000)]  # pull a batch, not the whole collection

    match = companies_col.find_one({
        "organisasjonsnummer": {"$in": no_data_orgs},
        "organisasjonsform.kode": organisasjonsform_kode
    })
    return match

as_example = find_no_data_example("AS")
enk_example = find_no_data_example("ENK")

print("=== AS example (no_data) ===")
if as_example:
    print(f"Org: {as_example['organisasjonsnummer']} | {as_example['navn']}")
    print(f"URL: {BASE_URL}{as_example['organisasjonsnummer']}")
else:
    print("None found in this batch - increase the .limit() above")

print("\n=== ENK example (no_data) ===")
if enk_example:
    print(f"Org: {enk_example['organisasjonsnummer']} | {enk_example['navn']}")
    print(f"URL: {BASE_URL}{enk_example['organisasjonsnummer']}")
else:
    print("None found in this batch - increase the .limit() above")

=== AS example (no_data) ===
Org: 834108852 | ADAM SNEKKERING AS
URL: https://data.brreg.no/regnskapsregisteret/regnskap/834108852

=== ENK example (no_data) ===
Org: 812213962 | ESPOSITO CONSULTING
URL: https://data.brreg.no/regnskapsregisteret/regnskap/812213962


**Output from the previous cell (recorded for reference):**

```
=== AS example (no_data) ===
Org: 834108852 | ADAM SNEKKERING AS
URL: https://data.brreg.no/regnskapsregisteret/regnskap/834108852

=== ENK example (no_data) ===
Org: 812213962 | ESPOSITO CONSULTING
URL: https://data.brreg.no/regnskapsregisteret/regnskap/812213962
```


In [9]:
import pymongo

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]

company = companies_col.find_one({"organisasjonsnummer": "834108852"})

print(f"Navn: {company.get('navn')}")
print(f"Organisasjonsform: {company.get('organisasjonsform')}")
print(f"Stiftelsesdato: {company.get('stiftelsesdato')}")
print(f"Registreringsdato Enhetsregisteret: {company.get('registreringsdatoEnhetsregisteret')}")
print(f"Registrert i Foretaksregisteret: {company.get('registrertIForetaksregisteret')}")
print(f"Registreringsdato Foretaksregisteret: {company.get('registreringsdatoForetaksregisteret')}")
print(f"Siste innsendte årsregnskap: {company.get('sisteInnsendteAarsregnskap')}")
print(f"Konkurs: {company.get('konkurs')}")
print(f"Under avvikling: {company.get('underAvvikling')}")
print(f"Registrert i MVA-registeret: {company.get('registrertIMvaregisteret')}")
print(f"Har registrert antall ansatte: {company.get('harRegistrertAntallAnsatte')}")

Navn: ADAM SNEKKERING AS
Organisasjonsform: {'links': [], 'kode': 'AS', 'beskrivelse': 'Aksjeselskap'}
Stiftelsesdato: 2024-08-03
Registreringsdato Enhetsregisteret: 2024-09-17
Registrert i Foretaksregisteret: True
Registreringsdato Foretaksregisteret: 2024-09-17
Siste innsendte årsregnskap: None
Konkurs: False
Under avvikling: False
Registrert i MVA-registeret: True
Har registrert antall ansatte: False


In [10]:
import pymongo
from collections import Counter

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"

# ============================================================
# OVERALL STATISTICS
# ============================================================
total_companies = companies_col.count_documents({})
total_fetched = financial_col.count_documents({})
total_success = financial_col.count_documents({"fetch_status": "success"})
total_no_data = financial_col.count_documents({"fetch_status": "no_data"})

print("=" * 60)
print("OVERALL STATISTICS (all organisasjonsform types)")
print("=" * 60)
print(f"Total companies in dataset: {total_companies:,}")
print(f"Total processed so far:     {total_fetched:,} ({total_fetched/total_companies*100:.1f}%)")
print(f"  - success:                {total_success:,} ({total_success/total_fetched*100:.1f}% of processed)")
print(f"  - no_data:                {total_no_data:,} ({total_no_data/total_fetched*100:.1f}% of processed)")

# ============================================================
# AS-SPECIFIC STATISTICS via server-side $lookup aggregation -
# avoids building a huge $in list (which caused DocumentTooLarge
# at ~1.17M org numbers, exceeding MongoDB's 16MB document limit)
# ============================================================
total_as_all = companies_col.count_documents({"organisasjonsform.kode": "AS"})

as_status_pipeline = [
    {"$lookup": {
        "from": "companies",
        "localField": "organisasjonsnummer",
        "foreignField": "organisasjonsnummer",
        "as": "company"
    }},
    {"$unwind": "$company"},
    {"$match": {"company.organisasjonsform.kode": "AS"}},
    {"$group": {"_id": "$fetch_status", "count": {"$sum": 1}}}
]
as_status_counts = {r["_id"]: r["count"] for r in financial_col.aggregate(as_status_pipeline)}
total_as_success = as_status_counts.get("success", 0)
total_as_no_data = as_status_counts.get("no_data", 0)
total_as_fetched = total_as_success + total_as_no_data

print(f"\n{'-'*60}")
print("AS (AKSJESELSKAP) STATISTICS")
print(f"{'-'*60}")
print(f"Total AS companies in dataset:  {total_as_all:,}")
print(f"AS companies processed so far:  {total_as_fetched:,} ({total_as_fetched/total_as_all*100:.1f}% of all AS)")
if total_as_fetched > 0:
    print(f"  - AS success:                  {total_as_success:,} ({total_as_success/total_as_fetched*100:.1f}% of AS processed)")
    print(f"  - AS no_data:                  {total_as_no_data:,} ({total_as_no_data/total_as_fetched*100:.1f}% of AS processed)")

# ============================================================
# Pull ALL AS no_data companies directly via aggregation -
# no giant $in array, and this is now the FULL population,
# not a sample
# ============================================================
as_no_data_pipeline = [
    {"$match": {"fetch_status": "no_data"}},
    {"$lookup": {
        "from": "companies",
        "localField": "organisasjonsnummer",
        "foreignField": "organisasjonsnummer",
        "as": "company"
    }},
    {"$unwind": "$company"},
    {"$match": {"company.organisasjonsform.kode": "AS"}},
    {"$project": {
        "_id": 0,
        "organisasjonsnummer": "$company.organisasjonsnummer",
        "navn": "$company.navn",
        "stiftelsesdato": "$company.stiftelsesdato",
        "registrertIForetaksregisteret": "$company.registrertIForetaksregisteret",
        "konkurs": "$company.konkurs",
        "underAvvikling": "$company.underAvvikling",
        "registrertIMvaregisteret": "$company.registrertIMvaregisteret"
    }}
]
as_no_data = list(financial_col.aggregate(as_no_data_pipeline))

print(f"\n{'-'*60}")
print(f"AS companies with no_data (FULL population, not a sample): {len(as_no_data):,}")
print(f"{'-'*60}")

# ============================================================
# Founding year breakdown
# ============================================================
founding_years = Counter()
no_founding_date = 0

for c in as_no_data:
    stiftet = c.get("stiftelsesdato")
    if stiftet:
        founding_years[int(stiftet[:4])] += 1
    else:
        no_founding_date += 1

print(f"\n--- Founding year breakdown (AS, no_data) ---")
for year in sorted(founding_years.keys(), reverse=True):
    count = founding_years[year]
    pct = count / len(as_no_data) * 100
    print(f"  {year}: {count} ({pct:.1f}%)")
if no_founding_date:
    print(f"  No founding date: {no_founding_date}")

founded_2024_plus = sum(c for y, c in founding_years.items() if y >= 2024)
founded_pre_2024 = sum(c for y, c in founding_years.items() if y < 2024)
print(f"\nFounded 2024 or later: {founded_2024_plus} ({founded_2024_plus/len(as_no_data)*100:.1f}%)")
print(f"Founded before 2024:   {founded_pre_2024} ({founded_pre_2024/len(as_no_data)*100:.1f}%)")

# ============================================================
# Examples founded BEFORE 2024 - now checking the FULL
# population, so this is a definitive answer, not sample-limited
# ============================================================
pre_2024_examples = [
    c for c in as_no_data
    if c.get("stiftelsesdato") and int(c["stiftelsesdato"][:4]) < 2024
]

print(f"\n=== Examples: AS, no_data, founded before 2024 (showing up to 5) ===")
if not pre_2024_examples:
    print("None found - ALL AS no_data cases (full population) were founded 2024 or later.")
for c in pre_2024_examples[:5]:
    org = c["organisasjonsnummer"]
    print(f"\n{c['navn']} (org {org})")
    print(f"  Founded:                      {c.get('stiftelsesdato')}")
    print(f"  Registrert i Foretaksregisteret: {c.get('registrertIForetaksregisteret')}")
    print(f"  Konkurs:                      {c.get('konkurs')}")
    print(f"  Under avvikling:              {c.get('underAvvikling')}")
    print(f"  Registrert i MVA-registeret:  {c.get('registrertIMvaregisteret')}")
    print(f"  URL: {BASE_URL}{org}")

OVERALL STATISTICS (all organisasjonsform types)
Total companies in dataset: 1,171,373
Total processed so far:     1,170,292 (99.9%)
  - success:                444,646 (38.0% of processed)
  - no_data:                725,646 (62.0% of processed)

------------------------------------------------------------
AS (AKSJESELSKAP) STATISTICS
------------------------------------------------------------
Total AS companies in dataset:  431,581
AS companies processed so far:  431,452 (100.0% of all AS)
  - AS success:                  403,782 (93.6% of AS processed)
  - AS no_data:                  27,670 (6.4% of AS processed)

------------------------------------------------------------
AS companies with no_data (FULL population, not a sample): 27,670
------------------------------------------------------------

--- Founding year breakdown (AS, no_data) ---
  2026: 21507 (77.7%)
  2025: 5288 (19.1%)
  2024: 679 (2.5%)
  2023: 153 (0.6%)
  2022: 20 (0.1%)
  2021: 4 (0.0%)
  2020: 2 (0.0%)
  201

In [11]:
import json
import os
import glob

DATA_DIR = "/home/jovyan/data"

# Locate the bulk download without assuming a filename.
candidates = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True))
big = [(p, os.path.getsize(p)) for p in candidates if os.path.getsize(p) > 50 * 1024**2]

print("JSON files found under %s:" % DATA_DIR)
for p in candidates:
    print("  %10.1f MB  %s" % (os.path.getsize(p) / 1024**2, p))

RAW_COMPANIES = big[0][0] if big else None
print("\nUsing:", RAW_COMPANIES)

# Framing check: is it really one array, or line-delimited? Read the ends only.
if RAW_COMPANIES:
    size = os.path.getsize(RAW_COMPANIES)
    with open(RAW_COMPANIES, "rb") as fh:
        head = fh.read(4096)
        fh.seek(max(0, size - 4096))
        tail = fh.read()
    print("\nfirst 200 bytes:", head[:200])
    print("last  200 bytes:", tail[-200:])
    print("\nnewlines in first 4 KB:", head.count(b"\n"))
    print("starts with '[':", head.lstrip()[:1] == b"[")
    print("ends with ']':  ", tail.rstrip()[-1:] == b"]")

JSON files found under /home/jovyan/data:
         0.0 MB  /home/jovyan/data/analytics_build_summary.json
         0.0 MB  /home/jovyan/data/benchmark_results.json
         0.0 MB  /home/jovyan/data/diagnose_variant_a.json
         0.0 MB  /home/jovyan/data/diagnostics_balance_and_layout.json
      1908.4 MB  /home/jovyan/data/enheter_alle.json
         0.0 MB  /home/jovyan/data/ndjson/_export_metadata.json
       174.3 MB  /home/jovyan/data/ndjson/financial_data/part-00000-2cd52ceb-7dac-4327-acd6-4d9304ab73ae-c000.json
       174.4 MB  /home/jovyan/data/ndjson/financial_data/part-00001-2cd52ceb-7dac-4327-acd6-4d9304ab73ae-c000.json
       174.6 MB  /home/jovyan/data/ndjson/financial_data/part-00002-2cd52ceb-7dac-4327-acd6-4d9304ab73ae-c000.json
       175.1 MB  /home/jovyan/data/ndjson/financial_data/part-00003-2cd52ceb-7dac-4327-acd6-4d9304ab73ae-c000.json
         0.0 MB  /home/jovyan/data/parquet/_export_metadata.json
         0.0 MB  /home/jovyan/data/profile_mongo.json
         0

In [12]:
import json
from collections import Counter, defaultdict

def stream_json_array(path, chunk_size=8 * 1024**2):
    """Yield elements of a top-level JSON array without loading the file."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as fh:
        buf = ""
        while True:                      # locate the opening bracket
            chunk = fh.read(chunk_size)
            if not chunk:
                return
            buf += chunk
            i = buf.find("[")
            if i != -1:
                buf = buf[i + 1:]
                break
        while True:
            buf = buf.lstrip()
            if buf[:1] == ",":
                buf = buf[1:].lstrip()
            if buf[:1] == "]":
                return
            if not buf:
                chunk = fh.read(chunk_size)
                if not chunk:
                    return
                buf += chunk
                continue
            try:
                obj, idx = dec.raw_decode(buf)
            except json.JSONDecodeError:
                chunk = fh.read(chunk_size)   # record spans the buffer edge
                if not chunk:
                    return
                buf += chunk
                continue
            yield obj
            buf = buf[idx:]


def typename(v):
    if v is None:
        return "null"
    if isinstance(v, bool):
        return "bool"
    if isinstance(v, int):
        return "int"
    if isinstance(v, float):
        return "float"
    if isinstance(v, str):
        return "string"
    if isinstance(v, list):
        return "array"
    if isinstance(v, dict):
        return "object"
    return type(v).__name__


class Profile:
    """Presence, observed types, emptiness and array length per dotted path."""

    def __init__(self):
        self.present = Counter()
        self.types = defaultdict(Counter)
        self.empty = Counter()
        self.max_len = Counter()
        self.n = 0

    def walk(self, obj, prefix=""):
        for k, v in obj.items():
            path = prefix + k
            self.present[path] += 1
            self.types[path][typename(v)] += 1
            if v in (None, "", [], {}):
                self.empty[path] += 1
            if isinstance(v, dict):
                self.walk(v, path + ".")
            elif isinstance(v, list):
                self.max_len[path] = max(self.max_len[path], len(v))
                for item in v:
                    self.types[path + "[]"][typename(item)] += 1
                    if isinstance(item, dict):
                        self.walk(item, path + "[].")

    def add(self, doc):
        self.n += 1
        self.walk(doc)

    def table(self):
        rows = []
        for path, cnt in self.present.most_common():
            rows.append({
                "path": path,
                "present": cnt,
                "pct": round(cnt / self.n * 100, 2),
                "types": dict(self.types[path]),
                "empty": self.empty[path],
                "max_array_len": self.max_len.get(path, 0) or None,
            })
        return rows


# Full pass. Expect several minutes on 2 GB.
prof_raw = Profile()
for i, doc in enumerate(stream_json_array(RAW_COMPANIES), 1):
    prof_raw.add(doc)
    if i % 200000 == 0:
        print("  %d records" % i)

print("\nTotal records in raw file: %d" % prof_raw.n)
print("Distinct paths: %d\n" % len(prof_raw.present))

print("%-58s %9s %7s  %s" % ("path", "present", "empty", "types"))
print("-" * 110)
for r in prof_raw.table():
    if r["path"].count(".") <= 1:          # top level plus one nesting level
        print("%-58s %9d %7d  %s" % (r["path"], r["present"], r["empty"], r["types"]))

with open(os.path.join(DATA_DIR, "profile_raw_companies.json"), "w") as fh:
    json.dump({"source": RAW_COMPANIES, "records": prof_raw.n,
               "paths": prof_raw.table()}, fh, indent=2)
print("\nWritten to data/profile_raw_companies.json")

  200000 records
  400000 records
  600000 records
  800000 records
  1000000 records

Total records in raw file: 1171373
Distinct paths: 112

path                                                         present   empty  types
--------------------------------------------------------------------------------------------------------------
links                                                        1171373 1171373  {'array': 1171373}
organisasjonsnummer                                          1171373       0  {'string': 1171373}
navn                                                         1171373       0  {'string': 1171373}
organisasjonsform                                            1171373       0  {'object': 1171373}
organisasjonsform.links                                      1171373 1171373  {'array': 1171373}
organisasjonsform.kode                                       1171373       0  {'string': 1171373}
organisasjonsform.beskrivelse                                1171373       0

In [13]:
import pymongo

client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["companiesdb"]

def profile_collection(name):
    p = Profile()
    # companies._id is a Mongo-assigned ObjectId and absent from the raw file.
    # financial_data._id is the organisasjonsnummer and must be profiled.
    drop_id = (name == "companies")
    for doc in db[name].find({}, no_cursor_timeout=False).batch_size(2000):
        if drop_id:
            doc.pop("_id", None)
        else:
            doc["_id"] = str(doc["_id"])   # ObjectId would break typename()
        p.add(doc)
        if p.n % 200000 == 0:
            print("  %s: %d" % (name, p.n))
    return p

profiles = {}
for name in ["companies", "financial_data"]:
    print("Profiling %s ..." % name)
    profiles[name] = profile_collection(name)
    print("  done, %d docs, %d paths\n" % (profiles[name].n, len(profiles[name].present)))

for name, p in profiles.items():
    print("\n=== %s (%d docs) ===" % (name, p.n))
    print("%-58s %9s %7s  %s" % ("path", "present", "empty", "types"))
    print("-" * 110)
    for r in p.table():
        if r["path"].count(".") <= 1:
            print("%-58s %9d %7d  %s" % (r["path"], r["present"], r["empty"], r["types"]))

# Does the raw file agree with what MongoDB holds?
raw_top = {k for k in prof_raw.present if "." not in k and "[]" not in k}
mongo_top = {k for k in profiles["companies"].present if "." not in k and "[]" not in k}
print("\nIn raw file only:  ", sorted(raw_top - mongo_top))
print("In MongoDB only:   ", sorted(mongo_top - raw_top))
print("Field count raw=%d mongo=%d" % (len(raw_top), len(mongo_top)))

with open(os.path.join(DATA_DIR, "profile_mongo.json"), "w") as fh:
    json.dump({name: {"docs": p.n, "paths": p.table()}
               for name, p in profiles.items()}, fh, indent=2)
print("\nWritten to data/profile_mongo.json")

Profiling companies ...
  companies: 200000
  companies: 400000
  companies: 600000
  companies: 800000
  companies: 1000000
  done, 1171373 docs, 112 paths

Profiling financial_data ...
  financial_data: 200000
  financial_data: 400000
  financial_data: 600000
  financial_data: 800000
  financial_data: 1000000
  done, 1170292 docs, 61 paths


=== companies (1171373 docs) ===
path                                                         present   empty  types
--------------------------------------------------------------------------------------------------------------
links                                                        1171373 1171373  {'array': 1171373}
organisasjonsnummer                                          1171373       0  {'string': 1171373}
navn                                                         1171373       0  {'string': 1171373}
organisasjonsform                                            1171373       0  {'object': 1171373}
organisasjonsform.links            

In [14]:
import json

with open("/home/jovyan/data/profile_mongo.json") as fh:
    prof = json.load(fh)

print("%-64s %9s %7s  %s" % ("path", "present", "empty", "types"))
print("-" * 118)
for r in prof["financial_data"]["paths"]:
    if r["path"].count(".") >= 2:
        print("%-64s %9d %7d  %s" % (r["path"], r["present"], r["empty"], r["types"]))

# Array multiplicity across both collections, which the first table omitted.
for name in ["companies", "financial_data"]:
    print("\n%s — array fields and observed max length:" % name)
    for r in prof[name]["paths"]:
        if r.get("max_array_len"):
            print("  %-56s %d" % (r["path"], r["max_array_len"]))

path                                                               present   empty  types
----------------------------------------------------------------------------------------------------------------------
data[].virksomhet.organisasjonsnummer                               445225       0  {'string': 445225}
data[].virksomhet.organisasjonsform                                 445225       0  {'string': 445225}
data[].virksomhet.morselskap                                        445225       0  {'bool': 445225}
data[].regnskapsperiode.fraDato                                     445225       0  {'string': 445225}
data[].regnskapsperiode.tilDato                                     445225       0  {'string': 445225}
data[].revisjon.ikkeRevidertAarsregnskap                            445225       0  {'bool': 445225}
data[].revisjon.fravalgRevisjon                                     445225       0  {'bool': 445225}
data[].regnkapsprinsipper.smaaForetak                               445225  